# MICAI 2026 · MLOps Workshop
## Notebook 3 of 3 · Model Monitoring and Data Drift
**Chihuahua, Mexico · November 2, 2026**

| | |
|---|---|
| **Workshop** | MICAI 2026 · MLOps Workshop: from raw data to a monitored model |
| **Dataset** | *Wine Quality* (white wine). P. Cortez, A. Cerdeira, F. Almeida, T. Matos and J. Reis (2009). *Modeling wine preferences by data mining from physicochemical properties.* Decision Support Systems 47(4), 547–553. Obtained from the UCI Machine Learning Repository. The file `modified.csv` is a deliberately corrupted copy prepared for this workshop. |
| **Tools** | Python, pandas, scikit-learn, SciPy, MLflow, Databricks and Google Colab |

**Declaration on the use of AI.** These materials were developed with the assistance of generative AI
(Anthropic's Claude, model Claude Opus 5.5), used to draft and review code, explanations and documentation.
The workshop instructors reviewed and validated all content and are responsible for it.

    ## What you will learn

    A model that scored well in notebook 2 can quietly fail weeks later, without a single error message, because the
    **data it receives has changed**. This is called **data drift**. In this notebook you will:

    1. Load the **production model** from MLflow, together with the data it was trained on.
    2. Learn two drift detectors: the **Kolmogorov–Smirnov test** and the **Population Stability Index (PSI)**.
    3. **Simulate drift** with a **Gaussian Mixture Model (GMM)**: a lab instrument that starts measuring wrong.
    4. Detect the drift, measure its **impact on the model**, and **log the monitoring report** to MLflow.

    **The workshop at a glance**

    | # | Notebook | Question it answers | MLflow experiment |
|---|---|---|---|
| 1 | Data quality, EDA and versioning | Can we trust the data? | `01_data_cleaning` |
| 2 | Training and experiment tracking | Which model, trained on which data version, works best? | `02_training` |
| 3 | **➡ Monitoring and data drift** | Is the model still valid once the data changes? | `03_drift_monitoring` |

    **How to use this notebook**
    - Run the cells from top to bottom: click a cell and press **Shift + Enter** (or use the ▶ button).
    - Every code cell is preceded by a text cell that explains what it does and why.
    - Sections marked **Deep dive (optional)** at the end are for later; they are not needed for the live session.

## Step 0 · Setup (about 1 minute)

Run the next cell. It shows two fields, or asks for them if you leave them empty:

1. **CONFIG_LINK**: paste the config link shown on the slide (or scanned from the QR code).
2. **NICKNAME**: any nickname for the class leaderboard. Please don't use your real name.

The cell then, automatically:
- downloads the workshop code and data from GitHub into this Colab machine,
- installs MLflow,
- downloads the workshop config, which contains the address of the shared MLflow server on Databricks and a temporary
  access key (the key is never printed),
- connects this notebook to the shared MLflow experiment `03_drift_monitoring`.

If anything fails (a mistyped link, the network, an expired key), the cell says so in one line and switches to a
**local** MLflow store inside Colab. The notebook still runs end to end; only the shared leaderboard is missing.

> Each notebook runs on its own Colab machine, so this setup is repeated at the top of every notebook,
> with the same link and nickname.

In [ ]:
#@title Step 0 · Workshop setup (click ▶ to run)
CONFIG_LINK = ""  #@param {type:"string"}
NICKNAME = ""  #@param {type:"string"}

import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/ITESM-MNA/MICAI_MLOps_2026.git"
try:
    import google.colab  # noqa: F401  (only available inside Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = Path("/content/MICAI_MLOps_2026")
    if not REPO_DIR.exists():
        print("Downloading the workshop files...")
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, str(REPO_DIR)], check=True)
    print("Installing MLflow (about 20 seconds)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements-colab.txt")], check=True)
else:  # running outside Colab, from the notebooks/ folder of the repository
    here = Path.cwd()
    REPO_DIR = Path(os.environ.get("MICAI_REPO_DIR", here.parent if (here.parent / "src").exists() else here))

sys.path.insert(0, str(REPO_DIR / "src"))
from micai_mlops.setup import setup_workshop
ws = setup_workshop(CONFIG_LINK, NICKNAME, notebook="03", repo_dir=REPO_DIR)

## 1. Load the production model and its reference data

The instructor published a **production model** in the training experiment (tag `role = reference`): a random forest
trained on the official `v1_clean_drop` version. Everyone monitors the same model, so your numbers will match the
ones on the screen. If you prefer, choose **my best run** to monitor your own best model from notebook 2.

Drift is always measured **against a reference**: the data the model was trained on. So we also load the official
datasets and rebuild that model's exact training set.

If the shared MLflow server is not available (offline mode), the cell trains the same production model locally in
about one second, from the same official data and with the same settings.

In [ ]:
#@title Model to monitor
MODEL_TO_MONITOR = "production (reference)"  #@param ["production (reference)", "my best run"]

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.metrics import (ConfusionMatrixDisplay, f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.mixture import GaussianMixture
import warnings

from micai_mlops import modeling
from micai_mlops.contract import FEATURES
from micai_mlops.official import load_official
from micai_mlops.splits import training_set, xy

sns.set_theme(style="whitegrid")
official, _ = load_official(ws)

query = ("tags.role = 'reference'" if MODEL_TO_MONITOR.startswith("production")
         else f"tags.alias = '{ws.alias}'")
runs = mlflow.search_runs(experiment_names=[ws.experiment("training")],
                          filter_string=f"{query} AND attributes.status = 'FINISHED'",
                          order_by=["metrics.roc_auc DESC"], max_results=1)
model, model_run_id = None, None
if not runs.empty:
    try:
        model_run_id = runs.iloc[0].run_id
        with warnings.catch_warnings():       # a model saved with another scikit-learn version
            warnings.simplefilter("ignore", InconsistentVersionWarning)
            model = mlflow.sklearn.load_model(f"runs:/{model_run_id}/model")
        data_version = runs.iloc[0]["tags.dataset_version"]
        print(f"✅ Loaded model from MLflow run {model_run_id[:8]}… ({runs.iloc[0]['tags.mlflow.runName']})")
    except Exception as exc:
        print(f"⚠️  Could not load the model from MLflow ({type(exc).__name__}).")
        model = None
if model is None:
    model, _, _, _ = modeling.train_reference(official)
    data_version, model_run_id = modeling.REFERENCE_VERSION, "local-reference"
    print("✅ Trained the production model locally (same data and settings as the published one).")

reference_df, _ = training_set(official[data_version], official["holdout"])
X_ref, y_ref = xy(reference_df, FEATURES)
X_new, y_new = xy(official["holdout"], FEATURES)
print(f"Reference data: {data_version} training set, {len(X_ref):,} wines")
print(f"Incoming batch: {len(X_new):,} new wines (the holdout, never seen by the model)")

## 2. How to detect drift, and a first check with no drift

For each feature we compare the **reference** distribution with the **incoming batch**:

- **Kolmogorov–Smirnov (KS) test:** the largest gap between the two cumulative distributions. A small p-value means
  the batch is unlikely to come from the same distribution. Because we test 11 features at once, we use the
  **Bonferroni correction**: a feature is significant only if p < 0.05 / 11.
- **Population Stability Index (PSI):** splits the reference into 10 bins and measures how much the share of data in
  each bin moved. Rule of thumb: **< 0.1** stable, **0.1–0.25** moderate shift, **> 0.25** major shift.

With hundreds of rows, KS can call even tiny, harmless differences "significant". So a feature is flagged as
**drifted** only when **both** agree: it is statistically significant (KS) **and** the shift is large enough to
matter (PSI ≥ 0.1).

First, a sanity check: the incoming batch comes from the same population as the training data, so nothing should be
flagged.

In [ ]:
ALPHA = 0.05
PSI_THRESHOLD = 0.1

def psi(reference, current, bins=10):
    """Population Stability Index, with bins taken from the reference quantiles."""
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_share = np.clip(np.histogram(reference, edges)[0] / len(reference), 1e-4, None)
    cur_share = np.clip(np.histogram(current, edges)[0] / len(current), 1e-4, None)
    return float(np.sum((cur_share - ref_share) * np.log(cur_share / ref_share)))

def drift_report(reference, current):
    rows = []
    for col in FEATURES:
        ks = stats.ks_2samp(reference[col], current[col])
        rows.append({"feature": col, "ks_statistic": ks.statistic, "ks_pvalue": ks.pvalue,
                     "psi": psi(reference[col], current[col])})
    report = pd.DataFrame(rows).set_index("feature")
    report["drift"] = (report["ks_pvalue"] < ALPHA / len(FEATURES)) & (report["psi"] >= PSI_THRESHOLD)
    return report.sort_values("psi", ascending=False)

def model_metrics(X, y):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {"roc_auc": roc_auc_score(y, proba), "f1": f1_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred),
            "share_predicted_good": pred.mean()}

report_no_drift = drift_report(X_ref, X_new)
print(f"Drifted features: {report_no_drift.index[report_no_drift['drift']].tolist() or 'none'}")
display(report_no_drift.round(4).head(5))
pd.Series(model_metrics(X_new, y_new)).round(3).to_frame("no drift")

## 3. Simulate drift with a Gaussian Mixture Model

**Scenario.** After maintenance, the lab's **density meter** is miscalibrated: it now reads higher than the true
density, and the error is larger for light wines than for heavy ones. The wines are the same and their true quality
has not changed; only the **measurement** has. In notebook 1 we saw that density is one of the features most related to
quality, so the model relies on it.

**Why a Gaussian Mixture Model?** The density of white wines is not one bell curve: it mixes several styles (light
and dry, medium, heavier and sweeter). A GMM describes the distribution as a sum of **3 Gaussian components**, each
with its own mean, spread and weight. We then:

1. fit the GMM on the reference densities and sort the components from lightest to heaviest;
2. **shift the mean of each component** (+1.5, +1.0 and +0.5 standard deviations, light to heavy);
3. move every incoming wine by the shift of the component it belongs to.

Each wine keeps all its other measurements and its true label, so any change in the model's results is caused by the
drift alone.

In [ ]:
DRIFT_FEATURE = "density"
SHIFTS_SD = [1.5, 1.0, 0.5]       # shift per component, light -> heavy, in reference standard deviations

def simulate_gmm_drift(reference_values, current_values, shifts_sd, seed=42):
    gmm = GaussianMixture(n_components=3, random_state=seed).fit(reference_values.to_numpy().reshape(-1, 1))
    order = np.argsort(gmm.means_.ravel())                     # components sorted light -> heavy
    shift = np.zeros(3)
    shift[order] = np.asarray(shifts_sd) * reference_values.std()
    component = gmm.predict(current_values.to_numpy().reshape(-1, 1))
    info = pd.DataFrame({"weight": gmm.weights_[order], "original mean": gmm.means_.ravel()[order],
                         "modified mean": gmm.means_.ravel()[order] + shift[order],
                         "shift (sd)": np.asarray(shifts_sd)},
                        index=["light", "medium", "heavy"])
    return current_values.to_numpy() + shift[component], info

drifted_values, gmm_info = simulate_gmm_drift(X_ref[DRIFT_FEATURE], X_new[DRIFT_FEATURE], SHIFTS_SD)
X_drift = X_new.copy()
X_drift[DRIFT_FEATURE] = drifted_values
display(gmm_info.round(5))

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.kdeplot(X_new[DRIFT_FEATURE], label="Incoming batch, correct meter", fill=True, color="#4c72b0", ax=ax)
sns.kdeplot(X_drift[DRIFT_FEATURE], label="Incoming batch, miscalibrated meter", fill=True, color="#c44e52", ax=ax)
ax.set(title="Density before and after the instrument drift", xlabel="density (g/cm³)", ylabel="density of wines")
ax.legend()
plt.show()

## 4. Detect the drift

The same report as in step 2, now on the batch measured with the miscalibrated meter.

In [ ]:
report_drift = drift_report(X_ref, X_drift)
print(f"Drifted features: {report_drift.index[report_drift['drift']].tolist() or 'none'}")
report_drift.round(4).head(5)

**How to read it.** Only `density` is flagged: its KS p-value is far below the Bonferroni threshold and its PSI is
well above 0.25 (a major shift). Every other feature stays stable. The detector found the problem **without needing
any labels**, which in production usually arrive weeks later, if ever.

## 5. What does the drift do to the model?
Here we *do* have the true labels of the batch, so we can measure the damage directly.

In [ ]:
impact = pd.DataFrame({"no drift": model_metrics(X_new, y_new),
                       "with drift": model_metrics(X_drift, y_new)})
impact["change"] = impact["with drift"] - impact["no drift"]
display(impact.round(3))

cm_fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, X) in zip(axes, [("No drift", X_new), ("With density drift", X_drift)]):
    ConfusionMatrixDisplay.from_predictions(y_new, model.predict(X), display_labels=["not good", "good"],
                                            ax=ax, colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()

**How to read it.** ROC-AUC barely moves: the model still *ranks* wines in almost the same order. But **recall drops
sharply**: higher density readings make wines look heavier, and heavier wines tend to be rated lower, so most good
wines now fall below the 0.5 decision threshold and are missed. No error is raised; the model simply makes worse
decisions. **This is why models need monitoring.**

## 6. Log the monitoring report to MLflow

The monitoring result is stored in the experiment `03_drift_monitoring`, linked to the model it checked
(`model_run_id`) and to the reference and incoming datasets. Over time, a series of such runs is the model's
**health record**.

In [ ]:
drifted = report_drift.index[report_drift["drift"]].tolist()
with mlflow.start_run(run_name=f"{ws.alias}-drift-{DRIFT_FEATURE}") as run:
    mlflow.set_tags({"alias": ws.alias, "model_run_id": model_run_id, "scenario": "density meter miscalibration",
                     "drift_detected": str(bool(drifted))})
    mlflow.log_params({"drift_feature": DRIFT_FEATURE, "shifts_sd": str(SHIFTS_SD), "gmm_components": 3,
                       "alpha_bonferroni": ALPHA / len(FEATURES), "psi_threshold": PSI_THRESHOLD,
                       "reference_version": data_version})
    mlflow.log_metrics({"n_drifted_features": len(drifted),
                        f"psi_{DRIFT_FEATURE}": report_drift.loc[DRIFT_FEATURE, "psi"],
                        f"ks_pvalue_{DRIFT_FEATURE}": report_drift.loc[DRIFT_FEATURE, "ks_pvalue"],
                        **{f"{k}_no_drift": v for k, v in impact["no drift"].items()},
                        **{f"{k}_with_drift": v for k, v in impact["with drift"].items()}})
    mlflow.log_input(mlflow.data.from_pandas(X_ref, name=f"{data_version}_train"), context="reference")
    mlflow.log_input(mlflow.data.from_pandas(X_drift, name="incoming_batch_drifted"), context="monitoring")
    mlflow.log_table(report_drift.reset_index(), "drift_report.json")
    mlflow.log_figure(cm_fig, "confusion_matrices.png")

print(f"✅ Logged monitoring run '{run.info.run_name}' in {ws.experiment('monitoring')}")
if ws.online:
    print("Open it in the MLflow UI:", ws.run_url("monitoring", run.info.run_id))

## 7. What would you do next?

Drift was detected and the model is making worse decisions. Options:

- **Fix the data, not the model.** Here the cause is a broken instrument: recalibrate it. Retraining the model on
  miscalibrated readings would teach it the error.
- **Retrain** when the world itself has changed (new grape varieties, a new region), using the new data as a new,
  versioned dataset, and compare the new model with the current one in MLflow before replacing it.
- **Automate the check:** run this report on every new batch and alert when a feature is flagged.

## Summary of the workshop

| Notebook | MLOps practice | Where it lives in MLflow |
|---|---|---|
| 1 | Test, clean, validate and **version** data | `01_data_cleaning`, official copy in `00_official_datasets` |
| 2 | **Track** experiments and compare models fairly | `02_training` |
| 3 | **Monitor** the model and detect drift | `03_drift_monitoring` |

Every model can be traced back to the exact data it was trained on, and every monitoring check to the model it
examined. That traceability is what makes machine learning reliable in production.

---
## Deep dive (optional): try another drift
Change the feature or the shifts, then re-run sections 3 to 5. For example, shift `alcohol` by
`[-1.0, -1.0, -1.0]`, or try a feature the model barely uses, such as `sulphates`, and see whether drift always
hurts the model.